# Phase 3 — Analyse

Cette phase a pour but de matérialiser les  problématiques de la boutique.

A savoir investiguer les données pour pouvoir faire ressortir les facteurs clés qui boostent le profit de l'entreprise

Et également les facteurs qui n'y contribuent pas.

le travail effectué servira à apporter des réponses concrètes aux questions business de **la phase zéro**

In [2]:
import pandas as pd
e_sales_clean = pd.read_csv ("../datasets/processed/online_retail_clean.csv")

### investigation sur chaque dimension
---

Il est beaucoup plus instinctif selon mon approche de comprendre ce que raconte les données concernés par la phase analyse

avant de rejoindre les métriques pour apporter des réponses concrètes aux problématiques business

### les produits (Description)

## Dictionnaire des métriques — Dimension Produits (Description)

| Métrique | Définition | Ce qu'elle mesure |
|---|---|---|
| CA (£) | Somme des TotalPrice par produit | Le revenu total généré par ce produit sur la période |
| Nb transactions | Nombre de lignes de commande (ventes + annulations) | nombre de lignes où ce produit apparait |
| Nb commandes | Nombre de factures distinctes contenant ce produit (InvoiceNo uniques) | Dans combien de commandes ce produit a-t-il été inclus |
| Dépense moyenne (£) | Moyenne des TotalPrice par ligne de commande | Le montant typique dépensé à chaque fois que ce produit est commandé |
| Taux annulation (%) | Annulations / Total transactions × 100 | La proportion de commandes de ce produit qui ont été annulées |

---

**Métrique globale (hors tableau produits)**

| Métrique | Définition | Ce qu'elle mesure |
|---|---|---|
| Valeur moyenne par commande (£) | Moyenne du TotalPrice par InvoiceNo | Le montant moyen d'une facture complète, tous produits confondus |
| Chiffre d'affaires global (£) | Somme des TotalPrice | Le revenu global de la boutique |

---

**Notes**
- CA  inclut les annulations (Quantity négative) — le CA est donc net des retours
- Dépense moyenne (£) — calculée sur les ventes uniquement (annulations exclues)
- Nb commandes mesure le reach (dans combien de commandes un produit apparaît),
  pas le volume d'unités vendues — un produit commandé en grande quantité une seule
  fois peut être sous-représenté dans ce classement.
- Valeur moyenne par commande est calculée sur les commandes non annulées uniquement
- Différence entre dépense moyenne et valeur moyenne:

---
### Dépense moyenne — montant moyen d'une transaction

Montant moyen dépensé à chaque fois qu'un produit apparaît dans une transaction.

> *En moyenne, chaque ligne du dataset vaut combien ?*

---
### Valeur moyenne par commande — montant moyen d'une facture

On regroupe d'abord toutes les lignes d'une même facture, on les somme,  
puis on fait la moyenne de ces sommes.

> *Quand un client passe une commande complète, il dépense combien au total ?*

---

In [2]:
# chiffre d'affaires 

print("chiffre d'affaires total par produit")
ca_par_produit = e_sales_clean.groupby('Description')['TotalPrice'].sum().nlargest(20)
print(f"{ca_par_produit} \n")


chiffre d'affaires total par produit
Description
DOTCOM POSTAGE                        206245.48
REGENCY CAKESTAND 3 TIER              164459.49
WHITE HANGING HEART T-LIGHT HOLDER     99612.42
PARTY BUNTING                          98243.88
JUMBO BAG RED RETROSPOT                92175.79
RABBIT NIGHT LIGHT                     66661.63
POSTAGE                                66230.64
PAPER CHAIN KIT 50'S CHRISTMAS         63715.24
ASSORTED COLOUR BIRD ORNAMENT          58792.42
CHILLI LIGHTS                          53746.66
SPOTTY BUNTING                         42030.67
JUMBO BAG PINK POLKADOT                41584.43
BLACK RECORD COVER FRAME               40578.21
PICNIC BASKET WICKER 60 PIECES         39619.50
SET OF 3 CAKE TINS PANTRY DESIGN       37378.79
DOORMAT KEEP CALM AND COME IN          36532.39
JAM MAKING SET WITH JARS               36069.34
WOOD BLACK BOARD ANT WHITE FINISH      35795.97
LUNCH BAG RED RETROSPOT                34717.66
POPCORN HOLDER                         

---
### constats
---
On se rend compte qu'il y a des noms qui apparaissent au niveau des descriptions

mais qui ne sont pas des produits même s'ils sont conformes aux patterns des noms de produits

On parle des valeurs comme `POSTAGE`, `AMAZON FEE`, `SAMPLES` qui correspondent plus à 

des frais de port, des ajustements comptables, ou autres frais qui n'ont rien à faire dans des 

analyses où on recense le chiffre d'affaires sur de véritables noms de produits

Ce premier résultat non filtré révèle des descriptions hors-produit en tête du classement

In [3]:
print((e_sales_clean['Description'] == 'DOTCOM POSTAGE').sum(), "frais de poste \n")
print((e_sales_clean['Description'] == 'POSTAGE').sum(), "pour la poste \n")
print((e_sales_clean['Description'] == 'AMAZON FEE').sum(), "frais de amazon \n")
print((e_sales_clean['Description'] == 'Manual').sum(), "manual \n")
print((e_sales_clean['Description'] == 'SAMPLES').sum(), "samples \n")

amazon = e_sales_clean[e_sales_clean['Description'] == 'AMAZON FEE']
manual = e_sales_clean[e_sales_clean['Description'] == 'Manual']
Samples = e_sales_clean[e_sales_clean['Description'] == 'SAMPLES']
postage = e_sales_clean[e_sales_clean['Description'] == 'POSTAGE']
dotcom = e_sales_clean[e_sales_clean['Description'] == 'DOTCOM POSTAGE']
print(amazon['InvoiceNo'].str.startswith('C').sum(), "annulations liés aux amazon fees \n")
print(manual['InvoiceNo'].str.startswith('C').sum(), "annulations liés à manual\n")
print(Samples['InvoiceNo'].str.startswith('C').sum(), "annulations liés à samples\n")
print(dotcom['InvoiceNo'].str.startswith('C').sum(), "annulations liés à dotcom postage\n")
print(postage['InvoiceNo'].str.startswith('C').sum(), "annulations liés à postage\n")

709 frais de poste 

1252 pour la poste 

34 frais de amazon 

567 manual 

62 samples 

32 annulations liés aux amazon fees 

244 annulations liés à manual

60 annulations liés à samples

1 annulations liés à dotcom postage

126 annulations liés à postage



---
### constats
---

en investigant de plus près, les valeurs de note de port, facture, ou ajustement comptables

sont liés à des annulations pour la plupart, donc on ne peut tout simplement pas les supprimer

Le mieux c'est de les isoler pour ne pas qu'ils interfèrent avec les métriques liés aux vrais produits.

In [3]:
descriptions_hors_produit = ['DOTCOM POSTAGE', 
                             'POSTAGE',
                             'SAMPLES',
                             'Manual', 
                             'AMAZON FEE']

# Base filtrée réutilisable
e_sales_produits = e_sales_clean[~e_sales_clean['Description'].isin(descriptions_hors_produit)]

#Base ventes uniquement: pour la dépense moyenne, le nombre de commandes
e_sales_ventes = e_sales_produits[~e_sales_produits['InvoiceNo'].str.startswith('C')]

# Calcul du taux d'annulation par produit
taux = e_sales_produits.groupby('Description').agg(
    total   =('InvoiceNo', 'count'),    # total : nombre de lignes (toutes transactions, ventes + annulations)
    annulees=('InvoiceNo', lambda x: x.str.startswith('C').sum())  # annulees : parmi ces lignes, celles dont l'InvoiceNo commence par 'C'
)                                                                  # le préfixe 'C' est la marque officielle d'annulation dans ce dataset

taux['taux_annulation'] = (taux['annulees'] / taux['total'] * 100).round(2)

# Tableau récapitulatif
recap = pd.DataFrame({
    'CA (£)'            : e_sales_produits.groupby('Description')['TotalPrice'].sum().round(2), # Combien rapporte un produit au total ?
    'Nb transactions'   : e_sales_produits.groupby('Description')['InvoiceNo'].count(),  # Est-ce que les produits X ou Y apparaissent souvent dans les transactions?
    'Nb commandes'      : e_sales_ventes.groupby('Description')['InvoiceNo'].nunique(), # ce produit se retrouve dans combien de commandes ?
    'Dépense moyenne(£)': e_sales_ventes.groupby('Description')['TotalPrice'].mean().round(2),  #montant moyen dépensé pour un produit X ou Y par transaction
    'Taux annulation(%)': taux['taux_annulation']  # A quelles proportions des commandes sont annulées ? 
                                                   # Quels produits présentent des taux élevés 
})

# Valeur moyenne par commande — métrique globale, pas par produit
valeur_par_commande = e_sales_ventes.groupby('InvoiceNo')['TotalPrice'].sum()
valeur_moyenne_commande = valeur_par_commande.mean().round(2)
print(f"Valeur moyenne par commande : {valeur_moyenne_commande} £\n")
print("-"*40)

# Chiffre d'affaires global
ca_global = e_sales_produits['TotalPrice'].sum()
print(f"chiffre d'affaires : {ca_global} £\n")
print("-"*40)
# Top 20 par CA
print(recap.sort_values('Dépense moyenne(£)', ascending=False).head(20).to_string())

Valeur moyenne par commande : 517.88 £

----------------------------------------
chiffre d'affaires : 9759115.16 £

----------------------------------------
                                      CA (£)  Nb transactions  Nb commandes  Dépense moyenne(£)  Taux annulation(%)
Description                                                                                                        
PAPER CRAFT , LITTLE BIRDIE             0.00                2           1.0           168469.60               50.00
PICNIC BASKET WICKER 60 PIECES      39619.50                2           2.0            19809.75                0.00
TEA TIME TEA TOWELS                  6045.00                2           2.0             3022.50                0.00
MISELTOE HEART WREATH CREAM           996.00                1           1.0              996.00                0.00
SET/5 RED SPOTTY LID GLASS BOWLS      734.40                1           1.0              734.40                0.00
WEEKEND BAG VINTAGE ROSE PAISLE



---
### constats 
---

`REGENCY CAKESTAND 3 TIER` est le produit le plus lucratif de la boutique avec **164459.49 £** 

mais par contraste, celui qui a le plus grand taux d'annulation parmi les produits les plus lucratifs avec un score de  **8.22 %**.

`WHITE HANGING HEART T-LIGHT HOLDER` qui est le plus commandé avec **2302 en termes de volume de commandes**, pour **2357 apparitions**

en termes de transactions. et c'est aussi le deuxième produit le plus lucratif en termes de chiffre d'affaires.

`PAPER CRAFT , LITTLE BIRDIE` affiche la dépense moyenne la plus élevée : **168 469.60£** sur une seule vente immédiatement annulée — CA net = 0£.

Le montant est vraisemblablement une erreur de saisie. Ce produit est donc exclu des constats commerciaux.

`PICNIC BASKET WICKER 60 PIECES` a pour montant moyen dépensé **19809.75 £**, et pourtant il est très peu commandé avec 

des ventes réellement finalisées.

L'analyse complète du taux d'annulation (hors Top 20 CA) révèle que 

les produits avec un **taux supérieur à 60%** concernent exclusivement 

des produits à 1 ou 2 transactions donc sans impact significatif sur le CA.


---
### les clients(CustomerID) et les zones géographiques (Country)
---

## Dictionnaire des métriques — Dimension Clients & Géographie

| Métrique | Définition | Ce qu'elle mesure |
|---|---|---|
| CA (£) | Somme des TotalPrice | Valeur d'achat totale — classe les clients et les pays par richesse |
| Nb commandes | Nombre de factures distinctes (InvoiceNo uniques, ventes + annulations) | Fréquence d'achat — identifie les clients et pays les plus actifs |

**Note**
- Nb commandes inclut les factures annulées (InvoiceNo commençant par 'C') — 
  ce choix mesure l'activité totale d'un client ou d'un pays, 
  ventes et retours confondus, pour contextualiser le taux d'annulation associé.
  Ce traitement diffère volontairement de la dimension Produits,
  où Nb commandes exclut les annulations pour mesurer la popularité réelle d'un produit.

In [5]:
e_sales_client = e_sales_produits

# Calcul du taux d'annulation par pays
taux_pays = e_sales_client.groupby('Country').agg(
    total   =('InvoiceNo', 'count'),    # total : nombre de lignes (toutes transactions, ventes + annulations)
    annulees=('InvoiceNo', lambda x: x.str.startswith('C').sum())  # annulees : parmi ces lignes, celles dont l'InvoiceNo commence par 'C'
)                                                                  # le préfixe 'C' est la marque officielle d'annulation dans ce dataset

taux_pays['taux_annulation'] = (taux_pays['annulees'] / taux_pays['total'] * 100).round(2)

# Calcul du taux d'annulation par client
taux_client = e_sales_client.groupby('CustomerID').agg(
    total   =('InvoiceNo', 'count'),    # total : nombre de lignes (toutes transactions, ventes + annulations)
    annulees=('InvoiceNo', lambda x: x.str.startswith('C').sum())  # annulees : parmi ces lignes, celles dont l'InvoiceNo commence par 'C'
)                                                                  # le préfixe 'C' est la marque officielle d'annulation dans ce dataset

taux_client['taux_annulation'] = (taux_client['annulees'] / taux_client['total'] * 100).round(2)

# Tableau récapitulatif
tableau_client = pd.DataFrame({
    'CA (£)'            : e_sales_client.groupby('CustomerID')['TotalPrice'].sum().round(2), 
    'Nb commandes'   : e_sales_client.groupby('CustomerID')['InvoiceNo'].nunique(),
    'Taux annulation(%)': taux_client['taux_annulation'] 
})
tableau_pays = pd.DataFrame({
    'CA (£)'            : e_sales_client.groupby('Country')['TotalPrice'].sum().round(2), 
    'Nb commandes'   : e_sales_client.groupby('Country')['InvoiceNo'].nunique(),
    'Nb transactions': e_sales_client.groupby('Country')['InvoiceNo'].count(),
    'Taux annulation(%)': taux_pays['taux_annulation'] 
})
print('Top 30 pays : \n')
print (tableau_pays.sort_values('CA (£)', ascending=False).head(30).to_string())
print("-"*48)
print('Top 20 client : \n')
print (tableau_client.sort_values('CA (£)', ascending=False).head(20).to_string())

Top 30 pays : 

                          CA (£)  Nb commandes  Nb transactions  Taux annulation(%)
Country                                                                            
United Kingdom        8264886.23         20910           485008                1.51
Netherlands            283479.54            97             2330                0.17
EIRE                   264186.92           349             8140                3.59
Germany                200526.54           577             9070                4.81
France                 182085.61           442             8200                1.61
Australia              136922.50            67             1256                5.81
Switzerland             52454.53            68             1958                1.69
Spain                   51679.57           100             2456                1.83
Belgium                 36573.22           117             1962                1.83
Japan                   35419.79            25              


---
### constats
---
La distribution du chiffre d'affaires est très asymétrique : `UK` tire les revenus et le nombre de commandes, en majorité.

Ils représentent **84.7% du CA global**, ce qui révèle une forte concentration des revenus sur le marché local..

L'écart avec le deuxième du classement, `Netherlands` , est très grand, **8 millions £** de différence en termes de chiffre.

D'affaires, et **20813 d'écart** en termes de nombre de commandes.

Les `USA` par contre ont **le plus grand taux d'annulation avec 38.62%** et figurent parmi les pays qui 

Commandent le moins, avec **7 commandes au total** et rapportent très peu en termes de chiffre d'affaires.

Les pays non spécifiés avec la valeur `Unspecified` arrivent en **24e position** en termes de chiffre d'affaires avec **4651.90 £**

La **21e position** en termes de nombre de commandes, sur **un total de 13**, je préfère ne pas les isoler.

Du côté des clients, la catégorie en tête de liste par  le chiffre d'affaires et le nombre de commandes , tous pays confondus, 

C’est celle sur laquelle on manque d’information : **les segments de clients inconnus**.

Ce segment est celui qui rapporte le plus, qui commande le plus, et qui représente un taux d'annulation très faible

Je ne peux pas les ignorer comme avec les pays `Unspecified`, la section des Clients `Unknown`

merite une investigation détaillée pour savoir quoi faire ensuite



---
### investigations spécifiques sur les clients inconnus
---

In [75]:
# Isoler les clients inconnus
unknown = e_sales_client[e_sales_client['CustomerID'] == 'Unknown']

# Panorama complet
panorama = unknown.groupby('Country').agg(
    ca_total         = ('TotalPrice', 'sum'),
    nb_commandes     = ('InvoiceNo', 'nunique'),
    nb_transactions  = ('InvoiceNo', 'count'),
    nb_produits      = ('Description', 'nunique')
).round(2)

# Proportions ajoutées après l'agg()
panorama['part_ca_%']           = (panorama['ca_total']/ tableau_pays['CA (£)']   * 100).round(2)
panorama['part_commandes_%']           = (panorama['nb_commandes'] / tableau_pays['Nb commandes']   * 100).round(2)
print(panorama.sort_values('ca_total', ascending=False).to_string())

                  ca_total  nb_commandes  nb_transactions  nb_produits  part_ca_%  part_commandes_%
Country                                                                                            
United Kingdom  1477622.37          1359           130148         3339      17.88              6.50
EIRE              12944.15            36              695          378       4.90             10.32
Hong Kong          9733.24             8              276          192     100.00            100.00
Unspecified        2080.17             5              201          150      44.72             38.46
Israel              913.57             3               47           46      11.56             33.33
France              691.06             3               66           62       0.38              0.68
Switzerland         623.65             3              117           87       1.19              4.41
Portugal            307.21             1               39           39       1.15              1.79



---
### constats
---

Les clients inconnus de **Hong Kong** et des pays Unspecified ont un taux important en termes de CA et  nombre de commandes

Ceux de **Hong Kong** sont tous des inconnus il représentent **100% de CA et des parts de commande**

Ceux des **pays inconnus**, représentent **44.72% de leur chiffre d'affaires total**, et **38.46 en part de commandes**


Le **Bahreïn**, par contre, c'est tout l'inverse : **le taux de CA est bien plus bas, c'est-à-dire nul**. Aucun chiffre d'affaires n'est généré 

par les  clients sont inconnus, mais ils représentent **la moitié des commandes.**

Après vérification rapide dans le dataset, les transactions associées aux clients inconnus sont effectivement au nombre de 2 pour le même produit

Une première transaction a été annulée en décembre 2010

et un deuxième transaction a eu lieu en janvier 2011, le même nom de produit, la même quantité, pour un totalPrice de 205.75 £


Les clients inconnus de **United Kingdom** représentent **17% environ en termes de chiffre d'affaires** 

et **6,5% en termes de nombre de commandes**. 


S'il faut se prononcer pour globalement:

Le segment Unknown domine le classement des clients en termes de CA. 

Puisque la problématique principale de la boutique est d'identifier les facteurs qui rapportent le plus, 

Ce segment mérite une attention particulière : qu'il s'agisse des 17% UK ou des 100% Hong Kong. 

La boutique devra décider comment identifier ou tracker ces clients anonymes

---
**Note méthodologique :** 

le taux d'annulation est calculé sur les lignes de transactions `(COUNT invoice_no)`. 

Pour les dimensions `client` et `pays`, un calcul sur les factures distinctes serait plus rigoureux — limite identifiée.

---
### les périodes temporelles
---

## Dictionnaire des métriques — Dimension Temporelle

| Métrique | Définition | Ce qu'elle mesure |
|---|---|---|
| CA (£) | Somme des TotalPrice par période | Revenu généré — identifie les périodes les plus lucratives |
| Nb transactions | Nombre de lignes par période | Volume d'activité brut — combien de produits ont été traités |
| Nb commandes | Nombre de factures distinctes (InvoiceNo uniques) par période | Fréquence de passage en caisse — combien de fois des clients ont commandé |

**Note**
- Nb commandes inclut les factures annulées — ce choix mesure 
  l'activité totale de la boutique sur chaque période,
  cohérent avec le traitement retenu pour les dimensions 
  Clients et Géographie.

In [15]:
e_sales_dates = e_sales_client.copy()

# Chiffre d'affaires total par période
ca_par_période = e_sales_dates.groupby(['year','month'])['TotalPrice'].sum().reset_index()
print(ca_par_période.sort_values('TotalPrice', ascending=False).to_string())

e_sales_dates['InvoiceDate'] = pd.to_datetime(e_sales_dates['InvoiceDate'])
print("-"*48)

for (year, month), group in e_sales_dates.groupby(['year', 'month']):
    if (year == 2010 and month == 12) or (year == 2011 and month == 12):
        dates = group['InvoiceDate'].dt.date
        print(f"{year}-{month:02d} | du {dates.min()} au {dates.max()} | {dates.nunique()} jours distincts")

    year  month  TotalPrice
11  2011     11  1425342.38
10  2011     10  1060418.33
9   2011      9  1010123.07
0   2010     12   757640.83
5   2011      5   730000.77
6   2011      6   723082.24
8   2011      8   700380.27
3   2011      3   679005.45
7   2011      7   675591.21
1   2011      1   578390.54
2   2011      2   498642.56
4   2011      4   480864.12
12  2011     12   439633.39
------------------------------------------------
2010-12 | du 2010-12-01 au 2010-12-23 | 20 jours distincts
2011-12 | du 2011-12-01 au 2011-12-09 | 8 jours distincts



---
### constats
---

la période des transactions se concentre principalement sur `l'année 2011`, ainsi que le mois de `décembre 2010`,

Mais ce mois n'est pas représentatif de toute `l'année 2010` et il n'est pas complet, allant du `1er au 23.`

En ce qui concerne `décembre 2011`, non seulement il n'est pas complet, il ne représente que le début du mois avec `8 jours`,

et rend donc son chiffre d'affaires biaisé, avec une différence entre le mois précédent, `novembre 2011` avec un CA de 

**1425342.38 £** et le mois suivant , `décembre 2011` avec un CA de **439633.39 £**.

La décision à prendre, **c'est d'exclure les 2 mois de décembre de l'analyse des dimensions temporelles.**

Les métriques concernées ne seront appliquées que pour la période de `janvier à novembre 2011.`

`décembre 2010` reste dans le dataset pour d'autres analyses, et sera exclus uniquement de la comparaison mensuelle temporelle. 

In [19]:
e_sales_dates_filtre = e_sales_dates[
    ~(
        (e_sales_dates['year'].isin([2010,2011])) &
        (e_sales_dates['month'] == 12)
    )
]

# Panorama complet
periode = e_sales_dates_filtre.groupby('month').agg(
    ca_total         = ('TotalPrice', 'sum'),
    nb_commandes     = ('InvoiceNo', 'nunique'),
    nb_transactions  = ('InvoiceNo', 'count'),
).round(2)

jour_ventes = e_sales_dates_filtre.groupby('day')['InvoiceNo'].nunique().reset_index()
jour_ventes.columns = ['jour_semaine', 'nb_ventes']

# Top 20 par CA
print(periode.sort_values('ca_total', ascending=False))
print("-"*48)
print(jour_ventes.sort_values('nb_ventes', ascending=False))

         ca_total  nb_commandes  nb_transactions
month                                           
11     1425342.38          3141            82462
10     1060418.33          2310            59241
9      1010123.07          2114            49303
5       730000.77          1935            36251
6       723082.24          1823            36112
8       700380.27          1580            34656
3       679005.45          1722            35882
7       675591.21          1686            38741
1       578390.54          1327            34519
2       498642.56          1282            27134
4       480864.12          1453            29197
------------------------------------------------
  jour_semaine  nb_ventes
3     Thursday       4302
5    Wednesday       3768
4      Tuesday       3742
1       Monday       3243
0       Friday       3192
2       Sunday       2126



---
### constats
---

Pour cette période de `2011`, la période de fin d'année est la plus lucrative en termes de génération de revenus et de commande

`septembre, octobre, et novembre`, domine le classement des CA, avec des sommes **dépassant 1 million de livres sterling**

Mais si on regarde le ratio transactions et commandes, il est très élevé peu importe le classement par chiffre d'affaires.

Pour le mois de `novembre`, c'est **82462 transactions** pour **3141 commandes**, soit **27 transactions par commande**.

Le mois `d'Avril` qui a le plus bas chiffre d'affaires suit ce schéma aussi, **avec 20 transactions par commande**

pour **1453 commandes et 29197 transactions.**

L'écart entre nb transactions et nb commandes est constant quel que soit le mois, 

suggérant que chaque commande regroupe en moyenne une vingtaine de lignes produits

Les commandes se concentrent en milieu de semaine, **jeudi en tête avec 4 302 ventes**, 

contre **2126 le dimanche , soit environ la moitié.**

---
# Synthèse d'analyse

Dans cette synthèse, on répond aux problématiques business de la boutique et on identifie les leviers de croissance du chiffre d'affaires.

---

## Analyse des revenus : qu'a-t-on découvert et quels KPI ont aidé à le faire ?

**C'est le CA par produit qui a révélé** que REGENCY CAKESTAND 3 TIER et WHITE HANGING HEART T-LIGHT HOLDER sont les produits qui rapportent le plus de revenus et qui se commandent le plus.

**C'est la dépense moyenne par produit qui a révélé** que PICNIC BASKET WICKER 60 PIECES est le deuxième produit avec la plus forte dépense moyenne par commande (19 809.75£), loin devant les autres, mais avec un volume de commandes très faible.

**C'est le CA par période temporelle qui a révélé** que les mois de fin d'année (de septembre à novembre) sont les plus lucratifs, avec un chiffre d'affaires atteignant le million et progressant au fil de ces mois.

**C'est le CA par pays qui a révélé** que le Royaume-Uni domine largement, aussi bien en revenus qu'en volume de commandes et de transactions.

**C'est le taux d'annulation par produit qui a été signalé : REGENCY CAKESTAND 3 TIER, malgré son CA élevé, présente le taux d'annulation le plus élevé parmi les produits lucratifs — un risque à surveiller.

---

## Comportement client : qu'a-t-on découvert et quels KPI ont aidé à le faire ?

**C'est le CA par client qui a révélé** que les clients inconnus (Unknown) génèrent le plus de revenus et sont les plus actifs en volume de commandes — mais sans information exploitable sur leur profil.

**C'est le taux d'annulation par client qui a révélé** que malgré leur volume important, les clients Unknown affichent un taux d'annulation de seulement 0.12% — ce qui nuance le risque lié à leur anonymat.

**La lecture combinée du CA et du nombre de commandes par client a révélé** que les profils qui commandent le plus ne sont pas nécessairement ceux qui rapportent le plus. , pour un cas concret, on a le client 14088 qui a 14 commandes avec 50415 £ générés, par contre le client 17841 a 167 commandes pour 39636 £ générés.
Certains clients commandent souvent de petits montants, d'autres rarement de montants élevés. 

---

## Tendances temporelles : qu'a-t-on découvert et quels KPI ont aidé à le faire ?

**C'est le CA par mois qui a révélé** que la saisonnalité influence fortement les revenus : les périodes de fin d'année (septembre–novembre) concentrent les pics, tandis que le début et le milieu d'année sont plus creux.

**C'est le nombre de transactions et de commandes par mois qui a révélé** que le ratio transactions/commandes reste stable toute l'année, autour de 20 transactions par commande en moyenne, donc la saisonnalité affecte le volume global, pas la structure des commandes.

**C'est le nombre de ventes par type de jour qui a révélé** que le jeudi génère le plus de ventes, mais le dimanche, les ventes baissent de moitié.

---

## Vérification des hypothèses de la phase zéro

### Les clients du Royaume-Uni peuvent être ceux qui effectuent le plus de transactions ?
**Réponse : OUI — hypothèse validée**

### Les périodes de fin d'année doivent être celles où le chiffre d'affaires est le plus élevé par rapport aux périodes plus creuses ?
**Réponse : OUI — hypothèse validée**

### Le taux d'annulation semble très faible par rapport aux achats finalisés ?
**Réponse : PLUS NUANCÉE**

Le taux global est effectivement faible, ce qui valide l'hypothèse en surface. Mais l'analyse par dimension révèle des signaux à surveiller :  

-  REGENCY CAKESTAND 3 TIER atteint 8.22% d'annulation parmi les produits les plus lucratifs, 
-  les États-Unis affichent un taux de 38.62% — anomalie majeure à l'échelle du pays
-  le client 15769 présente 11.56% d'annulations.

  Le taux global masque donc des concentrations de risque réelles sur certains produits, pays et clients spécifiques.

---

## Limites & points de vigilance — Synthèse Phase 3

### Anomalies dataset — impact sur la fiabilité
| # | Anomalie | Description | Impact |
|---|----------|-------------|--------|
| 1 | **Décembres tronqués** | décembre 2010 (23 jours) et décembre 2011 (8 jours) exclus de la comparaison mensuelle | Biais sur la mesure de saisonnalité — décembre non comparable aux autres mois |
| 2 | **Descriptions hors-produit** | POSTAGE, AMAZON FEE, SAMPLES, Manual isolés manuellement | Non scalable sans catégorisation dans le dataset — traitement au cas par cas |
| 3 | **Double incertitude Unknown / Unspecified** | 44.72% du CA du pays Unspecified provient de clients Unknown | Deux zones d'ombre sur le même flux financier — non exploitable stratégiquement |

### Limites d'analyse — corrigées en cours de projet
| # | Limite | Correction apportée |
|---|--------|---------------------|
| 1 | **Dépense moyenne produit incluait les annulations** | Recalculée sur `ventes_reeles` uniquement — les annulations faussaient la métrique |
| 2 | **`nb_commandes` utilisé à la place de `quantity`** pour les produits | Choix assumé et documenté : `nb_commandes` mesure le reach produit. Limite semi-corrigée — `quantity` reste la métrique de volume exacte |
| 3 | **Incohérence de définition du `nb_commandes`** entre dimensions | Dimension Produits : commandes abouties uniquement. Dimensions Clients, Géographie, temporel : toutes les commandes annulées incluses. Documenté mais non unifié |

### Limites d'analyse — identifiées, non corrigées
| # | Limite | Nuance |
|---|--------|--------|
| 1 | **Taux d'annulation calculé sur les lignes** (`COUNT invoice_no`) plutôt que sur les factures distinctes (`COUNT DISTINCT invoice_no`) | Sur les dimensions Clients, Géographie et Temporel, une facture multi-produits pèse autant de fois qu'elle a de lignes — surestimation possible sur les grandes commandes |
| 2 | **Taux d'annulation global non calculé** | Sans taux de référence global, les taux concentrés (USA 38%, REGENCY 8.22%) ne peuvent pas être mis en perspective — le masque statistique n'est pas levé |
| 3 | **Part du CA par produit non calculée** | La retransposition SQL révèle qu'aucun produit n'atteint 2% du CA global — forte dilution du catalogue. Piste non exploitée en Phase 3 |
| 4 | **Nombre de clients Unknown par pays non calculé** | L'investigation s'est concentrée sur l'impact CA des Unknown sans qualifier leur nombre. Hong Kong = 1 seul client Unknown avec 276 transactions — profil très différent d'un anonymat diffus |
| 5 | **Parts du CA géographique** | Calculées pour le UK uniquement. Une vue complète des parts par pays aurait renforcé l'analyse de concentration géographique |

### Note méthodologique — taux d'annulation
Le taux d'annulation est calculé sur les lignes de transactions (`COUNT invoice_no`) pour rester cohérent avec l'analyse Pandas. Un calcul sur les factures distinctes (`COUNT DISTINCT invoice_no`) serait plus rigoureux pour les dimensions Client et Géographie — piste d'amélioration identifiée 